# 04 - Joint Multi-Task Training (Classification + Detection + Segmentation)

Combines all three heads on **one shared encoder** and fine-tunes them together, alternating batches between the segmentation source (SIIM-ACR) and the detection source (RSNA) - exactly like the original MultiCheXNet training scheme (`MTL_generatot` in the Keras repo), reimplemented as `MTLJointLoader`.

We warm-start from the single-head checkpoints trained in notebooks 01-03 (optional but recommended - it converges much faster than starting from ImageNet alone).

In [ ]:
import os
import sys

REPO_NAME = "multi-task-chest-xray-analysis-system"
PROJECT_DIR = f"/kaggle/working/{REPO_NAME}"

assert os.path.exists(PROJECT_DIR), f"{PROJECT_DIR} does not exist!"

sys.path.insert(0, PROJECT_DIR)

print("Project directory:", PROJECT_DIR)

import torch
from src import config, engine
from src.models.mtl_model import MultiCheXNet
from src.datasets.siim_acr import SIIMACRDataset, build_siim_dataframe, get_default_train_augmentation as seg_aug, train_val_split as seg_split
from src.datasets.rsna import RSNADataset, load_rsna_dataframe, get_default_train_augmentation as det_aug, train_val_split as det_split, rsna_collate_fn
from src.datasets.mtl_dataset import MTLJointLoader
from src.utils.visualize import show_training_curves

print("device:", config.DEVICE)


## 1. Datasets (same paths as before)

In [ ]:
SEG_CSV_PATH = "/kaggle/input/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data/train-rle.csv"
SEG_IMG_PATH = "/kaggle/input/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data"
DET_CSV_PATH = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv"
DET_IMG_PATH = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images"


# only_positive=True for the segmentation source only, matching the
# upstream reference repo's default (coursat-ai/MultiCheXNet) - see
# notebook 01 for the full explanation. The joint loader already pulls
# segmentation and detection batches from two independent DataLoaders (see
# MTLJointLoader), so this only affects what the segmentation head is
# supervised on; the classifier/detector heads still see the full,
# unfiltered mix of positive and negative images as before.
seg_df = build_siim_dataframe(SEG_CSV_PATH, SEG_IMG_PATH, only_positive=True)
det_df = load_rsna_dataframe(DET_CSV_PATH)

seg_train_ids, seg_val_ids = seg_split(seg_df, val_fraction=0.15)
det_train_ids, det_val_ids = det_split(det_df, val_fraction=0.15)

seg_train_ds = SIIMACRDataset(seg_df, seg_train_ids, augmentation=seg_aug())
seg_val_ds   = SIIMACRDataset(seg_df, seg_val_ids)
det_train_ds = RSNADataset(det_df, det_train_ids, DET_IMG_PATH, augmentation=det_aug())
det_val_ds   = RSNADataset(det_df, det_val_ids, DET_IMG_PATH)

BATCH_SIZE = 8   # joint batches are a bit heavier (3 losses); 8 is safe on a T4
# num_workers=2 for throughput. You may see a harmless
# "AssertionError: can only test a child process" during worker cleanup
# under Kaggle/Jupyter - cosmetic only, doesn't affect training.
train_loader = MTLJointLoader(seg_train_ds, det_train_ds, batch_size=BATCH_SIZE,
                              num_workers=2, det_collate_fn=rsna_collate_fn)
val_loader = MTLJointLoader(seg_val_ds, det_val_ds, batch_size=BATCH_SIZE,
                            num_workers=2, shuffle=False, det_collate_fn=rsna_collate_fn)
print(len(train_loader), "train batches/epoch,", len(val_loader), "val batches/epoch")


## 2. Build the full 3-headed model

In [ ]:
model = MultiCheXNet(pretrained_encoder=True, add_classifier=True,
                     add_detector=True, add_segmenter=True).to(config.DEVICE)
print("Total params (M):", sum(p.numel() for p in model.parameters())/1e6)


## 3. (Optional but recommended) warm-start each head from its single-task checkpoint

Because every head + the shared encoder are just named `nn.Module`s, we can load each single-task checkpoint's `state_dict` and copy over only the matching keys (`strict=False`), instead of the manual layer-index surgery the original Keras code needed.

In [ ]:
def load_partial(model, ckpt_path):
    if not ckpt_path:
        return
    state = torch.load(ckpt_path, map_location=config.DEVICE)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"loaded {ckpt_path}: {len(state)-len(missing)} tensors matched, "
          f"{len(missing)} missing, {len(unexpected)} unexpected")

load_partial(model, "/kaggle/working/multi-task-chest-xray-analysis-system/segmenter_best.pt")
load_partial(model, "/kaggle/working/multi-task-chest-xray-analysis-system/detector_best.pt")
load_partial(model, "/kaggle/working/multi-task-chest-xray-analysis-system/classifier_best.pt")


## 4. Joint fine-tuning

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-5)
scaler = torch.amp.GradScaler('cuda', enabled=(config.DEVICE == "cuda"))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2)

# Same reasoning as notebook 01: give the scheduler (patience=2) room to
# drop the LR a couple of times and let early stopping (not a hardcoded
# epoch count) decide when training is actually done, rather than risking
# a cutoff mid-improvement.
EPOCHS = 30
LOSS_WEIGHTS = (1.0, 1.0, 1.0)   # (classification, detection, segmentation)

history = engine.fit(
    engine.train_one_epoch_mtl, engine.evaluate_mtl,
    model, train_loader, val_loader, optimizer, EPOCHS,
    device=config.DEVICE, checkpoint_path="/kaggle/working/multi-task-chest-xray-analysis-system/mtl_best.pt",
    monitor="loss", mode="min", scaler=scaler, loss_weights=LOSS_WEIGHTS,
    # max_grad_norm=5.0 (engine default) - guards against the same kind of
    # mid-training divergence spike observed in notebook 01 without it.
    scheduler=scheduler, patience=7, pos_weight="auto",
)


## 5. Training curves

In [ ]:
show_training_curves({k: v for k, v in history.items() if k.endswith('loss')}, figsize=(20,4))
show_training_curves({k: v for k, v in history.items() if 'acc' in k})
show_training_curves({k: v for k, v in history.items() if 'pos_dice' in k})


## 6. Qualitative check on a validation image from each source

In [ ]:
from src.utils.visualize import show_image_mask, show_image_boxes
from src.utils.bbox_utils import decode_predictions

model.load("/kaggle/working/multi-task-chest-xray-analysis-system/mtl_best.pt", map_location=config.DEVICE)
model.eval()

# segmentation example
img, mask, label = seg_val_ds[0]
with torch.no_grad():
    out = model(img.unsqueeze(0).to(config.DEVICE))
pred_mask = out["seg"][0,0].cpu().numpy()
pred_class = config.CLASS_NAMES[out["class"][0].argmax().item()]
show_image_mask(img.permute(1,2,0).numpy(), mask[0].numpy(), pred_mask>0.5, title=f"pred class: {pred_class}")

# detection example
idx = next(i for i in range(len(det_val_ds)) if len(det_val_ds[i][3])>0)
img, target, label, gt_boxes = det_val_ds[idx]
with torch.no_grad():
    out = model(img.unsqueeze(0).to(config.DEVICE))
preds = decode_predictions(out["det"][0], conf_threshold=0.3)
pred_class = config.CLASS_NAMES[out["class"][0].argmax().item()]
show_image_boxes(img.permute(1,2,0).numpy(), [p["box"] for p in preds], [p["score"] for p in preds],
                 title=f"pred class: {pred_class}")


Final joint checkpoint saved to `/content/mtl_best.pt`. This is the file you'll load in `05_Inference_and_Deployment.ipynb`.